# 🏆 Projeto Final - Camada Gold: Análise, Visualização e Dashboard
**Curso:** Análise de Dados com Python - SCtec  
**Aluno:** L. Felipe Martins  

Este Jupyter Notebook consolida a **Fase 3 (Camada Gold)** da nossa arquitetura de dados. 
Aqui realizamos:
1. Resolução das **7 Perguntas de Negócio** com exibições textuais e gráficas estáticas e defensivas.
2. Um **Dashboard Interativo Integrado (Tema Escuro - Estilo Power BI)** usando Plotly e `ipywidgets` para análise em tempo real.
3. Persistência da tabela física agregada `gold_resumo_orgaos` no PostgreSQL com confirmação transacional (`commit`).
4. Validação final lendo os dados de volta do banco de dados.

---
### ⚙️ 1. Gerenciamento Automatizado de Dependências e Carga de Dados

In [ ]:
import sys
import subprocess

# Lista de dependências críticas para o ambiente interativo do Jupyter
dependencias = ["plotly", "ipywidgets", "nbformat"]
instalou = False

for lib in dependencias:
    try:
        __import__(lib)
    except ImportError:
        print(f"[*] Instalação automática da biblioteca ausente: {lib}...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", lib, "--break-system-packages"
        ])
        instalou = True

if instalou:
    print("[SUCCESS] Dependências instaladas! Por favor, reinicie o Kernel do Jupyter para carregar as alterações.")
else:
    print("[+] Todas as dependências necessárias estão presentes no ambiente!")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from ipywidgets import interact
import warnings
from banco import conectar, executar, inserir_em_lote
import plotly.graph_objects as go
from IPython.display import HTML, display

warnings.filterwarnings('ignore')

# Configurações estéticas para os gráficos estáticos do Matplotlib
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'figure.titlesize': 16,
    'text.color': 'black',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black'
})

# Conexão com o PostgreSQL e carga dos dados refinados
conexao = conectar()
df_viagem = pd.read_sql_query("SELECT * FROM silver_viagem", conexao)
df_pagamento = pd.read_sql_query("SELECT * FROM silver_pagamento", conexao)
df_passagem = pd.read_sql_query("SELECT * FROM silver_passagem", conexao)
df_trecho = pd.read_sql_query("SELECT * FROM silver_trecho", conexao)
conexao.close()

print("\n[*] Dados da Camada Silver carregados com sucesso:")
print(f"  • Viagens: {len(df_viagem):,}")
print(f"  • Pagamentos: {len(df_pagamento):,}")
print(f"  • Passagens: {len(df_passagem):,}")
print(f"  • Trechos: {len(df_trecho):,}")

[+] Todas as dependências necessárias estão presentes no ambiente!

[*] Dados da Camada Silver carregados com sucesso:
  • Viagens: 341,860
  • Pagamentos: 606,916
  • Passagens: 167,260
  • Trechos: 290,259


---
### 💼 2. Resolução das Perguntas de Negócio

#### **P1. Quais são os 5 órgãos com maior custo total?**

In [ ]:
# Agrupamento e agregação (calculando também a volumetria de viagens para o Hover)
q1 = df_viagem.groupby('nome_orgao_superior').agg(
    valor_total=('valor_total', 'sum'),
    total_viagens=('id_viagem', 'count')
).reset_index()

# Ordenação decrescente e seleção dos 5 maiores
q1_top5 = q1.sort_values(by='valor_total', ascending=False).head(5)

# Exibição de controle textual no console
print("=== P1: OS 5 ÓRGÃOS COM MAIOR CUSTO TOTAL ===")
for idx, row in enumerate(q1_top5.itertuples(), 1):
    print(f"{idx}. {row.nome_orgao_superior:<50} | R$ {row.valor_total:,.2f} ({row.total_viagens:,} viagens)")

# Preparação do Gráfico (Invertido para o maior ficar no topo da exibição horizontal)
q1_grafico = q1_top5.sort_values(by='valor_total', ascending=True)

# Paleta de cores corporativas de alto contraste para fundo claro (da menor para a maior barra)
cores_premium_multi = [
    '#475569',  # Cinza Slate (5º colocado)
    '#b45309',  # Âmbar/Ouro Velho (4º colocado)
    '#4338ca',  # Índigo (3º colocado)
    '#0f766e',  # Verde Marinho/Teal (2º colocado)
    '#1e3a8a'   # Azul Escuro Corporativo (1º colocado)
]

fig_p1 = go.Figure()

fig_p1.add_trace(go.Bar(
    y=q1_grafico['nome_orgao_superior'],
    x=q1_grafico['valor_total'],
    orientation='h',
    marker=dict(
        color=cores_premium_multi,
        line=dict(color='#ffffff', width=1.5)  # Divisória branca limpa
    ),
    # Rótulos de dados externos formatados em Milhões (M) de R$ para clareza visual
    text=[f" <b>R$ {val / 1e6:.1f}M</b> " for val in q1_grafico['valor_total']],
    textposition='outside',
    textfont=dict(color='#1e293b', size=11, family="Segoe UI, Arial"),
    # Estruturação padronizada do Hover (Tooltip) seguindo o modelo da P2
    hovertemplate=(
        "<b>🏛️ Órgão:</b> %{y}<br>"
        "<b>💵 Gasto Total:</b> R$ %{customdata[0]:,.2f}<br>"
        "<b>✈️ Qtd. de Viagens:</b> %{customdata[1]:,}<extra></extra>"
    ),
    customdata=list(zip(q1_grafico['valor_total'], q1_grafico['total_viagens'])),
    width=0.55  # Espessura ideal para visualização executiva
))

# Estilização Padronizada do Hover (Mesmo padrão visual da P2)
fig_p1.update_traces(
    hoverlabel=dict(
        bgcolor="#1e293b",          # Fundo escuro elegante (Contraste com a página branca)
        bordercolor="#e2e8f0",      # Borda suave
        font=dict(
            color="#ffffff",        # Texto branco
            size=12,
            family="Segoe UI, Arial"
        ),
        align="left"
    )
)

# Layout Clean (Estilo Relatório Executivo)
fig_p1.update_layout(
    title=dict(
        text="<b>Top 5 Órgãos Superiores por Gasto Acumulado</b><br><span style='font-size: 11px; color: #64748b;'>Análise consolidada de custos totais de viagens (2025)</span>",
        font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"),
        x=0.02,
        y=0.92
    ),
    template="plotly_white",        # Força o template branco do Plotly
    paper_bgcolor="#ffffff",        # Fundo do papel
    plot_bgcolor="#ffffff",         # Fundo da área de plotagem
    height=380,                     # Altura otimizada para comportar 5 barras confortavelmente
    margin=dict(l=320, r=120, t=90, b=30),  # Margem esquerda generosa para evitar cortes nos ministérios
    showlegend=False,
    # Oculta o eixo X para evitar redundância (os valores em milhões já estão visíveis)
    xaxis=dict(
        visible=False,
        range=[0, q1_grafico['valor_total'].max() * 1.25]  # Margem de segurança à direita para o texto
    ),
    yaxis=dict(
        showgrid=False,
        showline=True,              # Linha guia suave do lado esquerdo
        linecolor="#e2e8f0",
        linewidth=1.5,
        tickfont=dict(size=11, color='#1e293b', family="Segoe UI, Arial", weight="bold")
    )
)

fig_p1.show()

=== P1: OS 5 ÓRGÃOS COM MAIOR CUSTO TOTAL ===
1. Ministério da Justiça e Segurança Pública          | R$ 486,933,121.65 (75,742 viagens)
2. Ministério da Defesa                               | R$ 156,070,304.49 (61,912 viagens)
3. Ministério da Educação                             | R$ 111,291,349.34 (65,295 viagens)
4. Ministério do Meio Ambiente e Mudança do Clima     | R$ 49,697,710.16 (19,413 viagens)
5. Ministério da Previdência Social                   | R$ 40,417,309.06 (8,190 viagens)


#### **P2. Quais são os 3 destinos com maior custo médio por viagem?**

In [ ]:
# Nota de Arquitetura para destaque metodológico aos avaliadores
html_nota_arquitetura = """
<div style="font-family: 'Segoe UI', Arial, sans-serif; max-width: 900px; margin-bottom: 20px;">
    <div style="background-color: #f0fdfa; border-left: 5px solid #0d9488; padding: 14px 16px; border-radius: 4px; box-shadow: 0 1px 2px rgba(0,0,0,0.05);">
        <strong style="color: #115e59; font-size: 13.5px; display: flex; align-items: center;">
            🧠 Nota de Engenharia de Dados: Decisão de Granularidade Geográfica
        </strong>
        <p style="color: #115e59; margin: 6px 0 0 0; font-size: 12.5px; line-height: 1.5; text-align: justify;">
            Diferente de abordagens baseadas estritamente na granularidade de <b>cidades isoladas</b>, este projeto consolidou a análise no nível de <b>Estado (UF) e País</b>. 
            Esta decisão foi tomada com base em duas premissas críticas de modelagem:
            <br><br>
            1. <b>Mitigação de Outliers de Rota:</b> Agrupar por cidades brutas introduz severa distorção estatística causada por itinerários multipontos complexos e específicos que ocorrem uma única vez (volume = 1). Uma única viagem cara contendo múltiplos destinos infla a média da cidade artificialmente, jogando registros irrelevantes para o topo do ranking.
            <br>
            2. <b>Consolidação Estatística (Visão Macro):</b> A unificação por Estado e País absorve a fragmentação das strings de viagens do Portal da Transparência, gerando <i>buckets</i> regionais limpos, consistentes e volumetricamente representativos para a tomada de decisão orçamentária.
        </p>
    </div>
</div>
"""
display(HTML(html_nota_arquitetura))

# Agrupamento utilizando a coluna física da Silver
q2 = df_viagem.groupby('destino_estado')['valor_total'].agg(['mean', 'count']).reset_index()

# Extração dinâmica dos dados de viagens Sigilosas para a Menção Honrosa
df_sigiloso = q2[q2['destino_estado'] == 'Sigiloso']
tem_sigiloso = not df_sigiloso.empty

if tem_sigiloso:
    media_sigiloso = df_sigiloso['mean'].values[0]
    qtd_sigiloso = df_sigiloso['count'].values[0]

# Filtro do Top 3 Destinos Geográficos Reais
q2_filtrado = q2[
    (q2['count'] >= 5) & 
    (~q2['destino_estado'].isin(["Sigiloso", "Não Informado", "Não informado"]))
].sort_values(by='mean', ascending=False).head(3)

# Exibição de controle textual no console
print("=== P2: OS 3 DESTINOS GEOGRÁFICOS COM MAIOR CUSTO MÉDIO ===")
for idx, row in enumerate(q2_filtrado.itertuples(), 1):
    print(f"{idx}. {row.destino_estado:<25} | Custo Médio: R$ {row.mean:,.2f} ({row.count} viagens)")
if tem_sigiloso:
    print(f"[*] Menção Honrosa: Sigiloso | Custo Médio: R$ {media_sigiloso:,.2f} ({qtd_sigiloso} viagens)")

# Construção do Gráfico de Linha Limpa (Fundo Branco)
q2_grafico = q2_filtrado.sort_values(by='mean', ascending=True)
cores_premium_claro = ['#2eb872', '#16824a', '#0a5c36']

fig_p2 = go.Figure()

fig_p2.add_trace(go.Bar(
    y=q2_grafico['destino_estado'],
    x=q2_grafico['mean'],
    orientation='h',
    marker=dict(
        color=cores_premium_claro,
        line=dict(color='#ffffff', width=1.5)
    ),
    text=[f" <b>R$ {val:,.2f}</b> " for val in q2_grafico['mean']],
    textposition='outside',
    textfont=dict(color='#0a5c36', size=12, family="Segoe UI, Arial"),
    hovertemplate=(
        "<b>📍 Destino:</b> %{y}<br>"
        "<b>💵 Custo Médio:</b> R$ %{x:,.2f}<br>"
        "<b>✈️ Qtd. de Viagens:</b> %{customdata}<extra></extra>"
    ),
    customdata=q2_grafico['count'],
    width=0.5
))

fig_p2.update_traces(
    hoverlabel=dict(
        bgcolor="#1e293b",
        bordercolor="#e2e8f0",
        font=dict(color="#ffffff", size=12, family="Segoe UI, Arial"),
        align="left"
    )
)

fig_p2.update_layout(
    title=dict(
        text="<b>Top 3 Destinos com Maior Custo Médio por Viagem</b><br><span style='font-size: 11px; color: #64748b;'>Análise focada em destinos geográficos (mínimo de 5 viagens)</span>",
        font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"),
        x=0.02,
        y=0.9
    ),
    template="plotly_white",
    paper_bgcolor="#ffffff",
    plot_bgcolor="#ffffff",
    height=280,
    margin=dict(l=130, r=120, t=80, b=10),
    showlegend=False,
    xaxis=dict(
        visible=False,
        range=[0, q2_grafico['mean'].max() * 1.25]
    ),
    yaxis=dict(
        showgrid=False,
        showline=True,
        linecolor="#e2e8f0",
        linewidth=1.5,
        tickfont=dict(size=13, color='#1e293b', family="Segoe UI, Arial", weight="bold")
    )
)

# Renderiza o gráfico
fig_p2.show()

# Inserção Visual da Menção Honrosa em HTML Padronizado
if tem_sigiloso:
    html_mencao = f"""
    <div style="font-family: 'Segoe UI', Arial, sans-serif; max-width: 900px; margin-top: 5px;">
        <div style="background-color: #f8fafc; border-left: 5px solid #475569; padding: 12px 16px; border-radius: 4px; box-shadow: 0 1px 2px rgba(0,0,0,0.05);">
            <strong style="color: #334155; font-size: 13.5px; display: flex; align-items: center;">
                🪪 Menção Honrosa de Auditoria: Operações de Caráter Sigiloso
            </strong>
            <p style="color: #475569; margin: 4px 0 0 0; font-size: 12.5px; line-height: 1.5;">
                Fora do escopo geográfico mapeado acima, a base de dados possui <b>{qtd_sigiloso:,}</b> viagens classificadas como 
                <span style="color: #1e293b; font-weight: bold;">"Informações protegidas por sigilo"</span>. 
                Estas missões especiais de segurança nacional apresentam um custo médio expressivo de 
                <span style="color: #0f766e; font-weight: bold;">R$ {media_sigiloso:,.2f}</span> por ocorrência.
            </p>
        </div>
    </div>
    """
    display(HTML(html_mencao))

=== P2: OS 3 DESTINOS GEOGRÁFICOS COM MAIOR CUSTO MÉDIO ===
1. Hong Kong                 | Custo Médio: R$ 38,520.17 (14 viagens)
2. Argélia                   | Custo Médio: R$ 30,449.45 (9 viagens)
3. Índia                     | Custo Médio: R$ 30,065.12 (141 viagens)
[*] Menção Honrosa: Sigiloso | Custo Médio: R$ 3,863.47 (51601 viagens)


#### **P3. Qual é a viagem de maior duração e seu custo total?**

In [ ]:
# Filtra viagens com duração preenchida
df_duracao = df_viagem[df_viagem['duracao_dias'].notnull()].copy()

# Identifica o recorde absoluto (possível anomalia de custo zerado)
viagem_absoluta = df_duracao.sort_values(by='duracao_dias', ascending=False).iloc[0]

# Filtra e identifica o recorde operacional (registro válido com custo real e maior que zero)
df_operacionais = df_duracao[df_duracao['valor_total'] > 0]
viagem_operacional = None
if not df_operacionais.empty:
    viagem_operacional = df_operacionais.sort_values(by='duracao_dias', ascending=False).iloc[0]

# Exibição textual de auditoria no console
print("=== P3: ANÁLISE DE DURAÇÃO DE VIAGENS ===")
print(f"• Anomalia Detectada:  ID {viagem_absoluta['id_viagem']} | Duração: {int(viagem_absoluta['duracao_dias'])} dias | Custo: R$ {viagem_absoluta['valor_total']:,.2f}")
if viagem_operacional is not None:
    print(f"• Recorde Operacional: ID {viagem_operacional['id_viagem']} | Duração: {int(viagem_operacional['duracao_dias'])} dias | Custo: R$ {viagem_operacional['valor_total']:,.2f}")

# Construção de Interface Visual unificada com o padrão Clean (Fundo Claro)
html_p3 = f"""
<div style="font-family: 'Segoe UI', Arial, sans-serif; margin-top: 15px; max-width: 900px;">
    
    <!-- CARD 1: ALERTA DE QUALIDADE DE DADOS (OUTLIER) -->
    <div style="background-color: #fef2f2; border-left: 5px solid #ef4444; padding: 16px; border-radius: 6px; margin-bottom: 16px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
        <strong style="color: #991b1b; font-size: 14px; display: flex; align-items: center;">
            ⚠️ Alerta de Qualidade de Dados (Outlier de Sistema)
        </strong>
        <p style="color: #7f1d1d; margin: 6px 0 0 0; font-size: 12.5px; line-height: 1.5;">
            A viagem de maior duração absoluta registrada possui <b>{int(viagem_absoluta['duracao_dias'])} dias</b> (ID: {viagem_absoluta['id_viagem']}), 
            porém apresenta custo zerado (<b>R$ {viagem_absoluta['valor_total']:,.2f}</b>) e viajante ausente. 
            Este registro foi isolado por representar uma provável anomalia ou cancelamento não expurgado na base original.
        </p>
    </div>
"""

if viagem_operacional is not None:
    # Formatação de datas para exibição visual limpa
    data_ini_fmt = pd.to_datetime(viagem_operacional['data_inicio']).strftime('%d/%m/%Y')
    data_fim_fmt = pd.to_datetime(viagem_operacional['data_fim']).strftime('%d/%m/%Y')
    
    html_p3 += f"""
    <!-- CARD 2: RECORDE OPERACIONAL REAL -->
    <div style="background-color: #f0fdf4; border-left: 5px solid #16a34a; padding: 18px; border-radius: 6px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
        <strong style="color: #14532d; font-size: 15px; display: flex; align-items: center;">
            🏆 Recorde Operacional Válido (Maior Viagem com Custo Consolidado)
        </strong>
        <div style="display: flex; justify-content: space-between; margin-top: 14px; color: #1e293b; font-size: 13px; line-height: 1.6;">
            <div style="flex: 1; padding-right: 20px;">
                <span style="color: #64748b; font-size: 11px; text-transform: uppercase; font-weight: bold;">Informações do Viajante</span><br>
                <b>👤 Viajante:</b> {viagem_operacional['nome_viajante'] if viagem_operacional['nome_viajante'] else 'Não Identificado'}<br>
                <b>🏛️ Órgão:</b> {viagem_operacional['nome_orgao_superior']}<br>
                <b>📍 Destino Consolidado:</b> {viagem_operacional['destino_estado']}
            </div>
            <div style="flex: 1; border-left: 1px solid #bbf7d0; padding-left: 24px;">
                <span style="color: #64748b; font-size: 11px; text-transform: uppercase; font-weight: bold;">Métricas do Período</span><br>
                <b>📅 Período Real:</b> {data_ini_fmt} até {data_fim_fmt}<br>
                <b>⏱️ Duração Efetiva:</b> <span style="color: #14532d; font-weight: bold; font-size: 14px;">{int(viagem_operacional['duracao_dias'])} dias</span><br>
                <b>💵 Custo Consolidado:</b> <span style="color: #14532d; font-weight: bold; font-size: 14px;">R$ {viagem_operacional['valor_total']:,.2f}</span>
            </div>
        </div>
    </div>
    """

html_p3 += "</div>"
display(HTML(html_p3))

=== P3: ANÁLISE DE DURAÇÃO DE VIAGENS ===
• Anomalia Detectada:  ID 0000000000020699856 | Duração: 383 dias | Custo: R$ 0.00
• Recorde Operacional: ID 0000000000020793594 | Duração: 378 dias | Custo: R$ 120,650.00


#### **P4. Qual o tipo de pagamento com maior valor médio?**

In [30]:
# 1. Agrupamento e cálculo da média e volumetria (count) para enriquecer o Hover
q4 = df_pagamento.groupby('tipo_pagamento')['valor'].agg(['mean', 'count']).reset_index()

# Exibição de controle textual no console
print("=== P4: RANKING DE TIPOS DE PAGAMENTO POR VALOR MÉDIO ===")
if not q4.empty:
    q4_ordenado = q4.sort_values(by='mean', ascending=False)
    for idx, row in enumerate(q4_ordenado.itertuples(), 1):
        sufixo = "🏆 [DESTAQUE]" if idx == 1 else ""
        print(f"{idx}. {row.tipo_pagamento:<35} | Valor Médio: R$ {row.mean:,.2f} ({row.count:,} lançamentos) {sufixo}")
else:
    print("[!] Tabela de pagamentos vazia ou sem dados válidos.")

# 2. Preparação do Gráfico (Invertido para o maior custo médio ficar no topo)
q4_grafico = q4.sort_values(by='mean', ascending=True)

# Aplicação da sistemática de destaque (P5/P6): Neutro para todas as modalidades, exceto a líder (#1)
cores_financeiras = ['#cbd5e1'] * (len(q4_grafico) - 1) + ['#1e3a8a']

# Define a cor do texto do rótulo de dados acompanhando a barra
cores_texto = ['#475569'] * (len(q4_grafico) - 1) + ['#1e3a8a']

fig_p4 = go.Figure()

fig_p4.add_trace(go.Bar(
    y=q4_grafico['tipo_pagamento'],
    x=q4_grafico['mean'],
    orientation='h',
    marker=dict(
        color=cores_financeiras,
        line=dict(color='#ffffff', width=1.5)  # Divisória branca limpa
    ),
    # Rótulos externos dinâmicos com cores coordenadas e formatação monetária
    text=[f" <b>R$ {val:,.2f}</b> " for val in q4_grafico['mean']],
    textposition='outside',
    textfont=dict(color=cores_texto, size=11, family="Segoe UI, Arial"),
    # Estruturação padronizada da Tooltip (Hover) em total sincronia com o projeto
    hovertemplate=(
        "<b>💳 Tipo de Pagamento:</b> %{y}<br>"
        "<b>💵 Valor Médio:</b> R$ %{x:,.2f}<br>"
        "<b>📊 Qtd. de Lançamentos:</b> %{customdata:,}<extra></extra>"
    ),
    customdata=q4_grafico['count'],
    width=0.6  # Proporção elegante de largura de barra adaptada ao layout
))

# 3. Estilização Padronizada do Hover (Mesmo "cartão flutuante" escuro das demais perguntas)
fig_p4.update_traces(
    hoverlabel=dict(
        bgcolor="#1e293b",          # Fundo grafite escuro
        bordercolor="#e2e8f0",      # Borda sutil cinza
        font=dict(
            color="#ffffff",        # Texto branco legível
            size=12,
            family="Segoe UI, Arial"
        ),
        align="left"
    )
)

# 4. Layout Clean & Executive (Fundo Claro de Alta Fidelidade)
fig_p4.update_layout(
    title=dict(
        text="<b>Tipos de Pagamento por Valor Médio Lançado</b><br><span style='font-size: 11px; color: #64748b;'>Análise comparativa com destaque estratégico na modalidade de maior desembolso médio</span>",
        font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"),
        x=0.02,
        y=0.92
    ),
    template="plotly_white",        # Força o fundo branco plano
    paper_bgcolor="#ffffff",
    plot_bgcolor="#ffffff",
    height=300,                     # Altura expandida para prover respiro ideal entre as barras
    margin=dict(l=160, r=130, t=85, b=25),
    showlegend=False,
    # Oculta o eixo X para eliminar linhas de grade e redundâncias numéricas
    xaxis=dict(
        visible=False,
        range=[0, q4_grafico['mean'].max() * 1.3]  # Margem de segurança para o texto externo
    ),
    yaxis=dict(
        showgrid=False,
        showline=True,              # Linha guia suave cinza na borda esquerda
        linecolor="#e2e8f0",
        linewidth=1.5,
        tickfont=dict(size=12, color='#1e293b', family="Segoe UI, Arial", weight="bold")
    )
)

fig_p4.show()

=== P4: RANKING DE TIPOS DE PAGAMENTO POR VALOR MÉDIO ===
1. DIÁRIAS                             | Valor Médio: R$ 2,078.28 (401,463 lançamentos) 🏆 [DESTAQUE]
2. PASSAGEM                            | Valor Médio: R$ 1,878.34 (188,985 lançamentos) 
3. Serviço correlato: seguro           | Valor Médio: R$ 447.51 (4,894 lançamentos) 
4. RESTITUIÇÃO                         | Valor Médio: R$ 245.70 (11,574 lançamentos) 


#### **P5. Qual o meio de transporte mais usado nos trechos?**

In [29]:
# Tratamento e agrupamento dos dados (Conforme sua lógica original)
df_transporte_valido = df_trecho[~df_trecho['meio_transporte'].isin([None, 'Sem informação', ''])]
q5 = df_transporte_valido['meio_transporte'].value_counts().reset_index()
q5.columns = ['meio_transporte', 'quantidade']

# Exibição de controle textual no console
print("=== P5: MEIO DE TRANSPORTE MAIS USADO NOS TRECHOS ===")
for idx, row in enumerate(q5.itertuples(), 1):
    print(f"{idx}. {row.meio_transporte:<20} | {row.quantidade:,} trechos")

# Preparação do Gráfico (Invertido para o mais utilizado ficar no topo)
q5_grafico = q5.sort_values(by='quantidade', ascending=True)

# Aplicação da sistemática da P6: Neutro para todos os modais, exceto o #1 (último da lista)
cores_logistica = ['#cbd5e1'] * (len(q5_grafico) - 1) + ['#1e3a8a']

# Define a cor do texto do rótulo de dados acompanhando a barra
cores_texto = ['#475569'] * (len(q5_grafico) - 1) + ['#1e3a8a']

fig_p5 = go.Figure()

fig_p5.add_trace(go.Bar(
    y=q5_grafico['meio_transporte'],
    x=q5_grafico['quantidade'],
    orientation='h',
    marker=dict(
        color=cores_logistica,
        line=dict(color='#ffffff', width=1.5)  # Divisória branca limpa
    ),
    # Rótulos externos dinâmicos com cores coordenadas
    text=[f" <b>{val:,}</b> " for val in q5_grafico['quantidade']],
    textposition='outside',
    textfont=dict(color=cores_texto, size=11, family="Segoe UI, Arial"),
    # Estruturação padronizada da Tooltip (Hover) em sincronia com o projeto
    hovertemplate=(
        "<b>🚀 Modal:</b> %{y}<br>"
        "<b>📊 Total de Trechos:</b> %{x:,}<extra></extra>"
    ),
    width=0.62  # Ajuste fino na largura para manter simetria com a P6
))

# Estilização Padronizada do Hover (Mesmo "cartão flutuante" escuro das demais perguntas)
fig_p5.update_traces(
    hoverlabel=dict(
        bgcolor="#1e293b",          # Fundo grafite escuro
        bordercolor="#e2e8f0",      # Borda sutil cinza
        font=dict(
            color="#ffffff",        # Texto branco legível
            size=12,
            family="Segoe UI, Arial"
        ),
        align="left"
    )
)

# Layout Clean & Executive (Garantia de conformidade visual para a entrega)
fig_p5.update_layout(
    title=dict(
        text="<b>Distribuição dos Meios de Transporte nos Trechos</b><br><span style='font-size: 11px; color: #64748b;'>Análise de volumetria absoluta com destaque cirúrgico no principal modal logístico</span>",
        font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"),
        x=0.02,
        y=0.92
    ),
    template="plotly_white",        # Força o fundo branco plano
    paper_bgcolor="#ffffff",
    plot_bgcolor="#ffffff",
    height=420,                     # Ajustado para 420px para manter simetria perfeita com a P6
    margin=dict(l=150, r=100, t=85, b=25),
    showlegend=False,
    # Oculta o eixo X para eliminar linhas de grade e redundâncias numéricas
    xaxis=dict(
        visible=False,
        range=[0, q5_grafico['quantidade'].max() * 1.2]  # Margem de segurança para o texto externo
    ),
    yaxis=dict(
        showgrid=False,
        showline=True,              # Linha guia suave cinza na borda esquerda
        linecolor="#e2e8f0",
        linewidth=1.5,
        tickfont=dict(size=12, color='#1e293b', family="Segoe UI, Arial", weight="bold")
    )
)

fig_p5.show()

=== P5: MEIO DE TRANSPORTE MAIS USADO NOS TRECHOS ===
1. Veículo Oficial      | 143,177 trechos
2. Aéreo                | 101,034 trechos
3. Rodoviário           | 23,119 trechos
4. Veículo Próprio      | 15,393 trechos
5. Inválido             | 3,883 trechos
6. Fluvial              | 3,268 trechos
7. Ferroviário          | 237 trechos
8. Marítimo             | 148 trechos


#### **P6. Qual UF de destino aparece em mais trechos?**

In [28]:
# Tratamento e agrupamento dos dados (Isolando nulos e vazios)
df_uf_valida = df_trecho[~df_trecho['destino_uf'].isin([None, 'Sem informação', ''])]

print("=== P6: RANKING DE UF DE DESTINO MAIS FREQUENTE NOS TRECHOS ===")
if not df_uf_valida.empty:
    q6 = df_uf_valida['destino_uf'].value_counts().reset_index()
    q6.columns = ['uf_destino', 'total_trechos']
    
    # Isola o Top 10 para uma visualização limpa e focada
    q6_top10 = q6.head(10)
    
    for idx, row in enumerate(q6_top10.itertuples(), 1):
        sufixo = "🏆 [DESTAQUE]" if idx == 1 else ""
        print(f"{idx:2d}. {row.uf_destino:<25} | {row.total_trechos:,} trechos {sufixo}")
else:
    print("[!] Nenhum trecho com UF de destino válida cadastrado.")

# Preparação do Gráfico (Invertido para o 1º colocado ficar no topo da tela)
q6_grafico = q6_top10.sort_values(by='total_trechos', ascending=True)

# Definição dinâmica da paleta: Todos os estados recebem cinza neutro, exceto o #1 (último da lista invertida)
# que recebe o Azul Marinho Corporativo de alto destaque
cores_ranking = ['#cbd5e1'] * (len(q6_grafico) - 1) + ['#1e3a8a']

# Define a cor do texto do rótulo de dados (Acompanha o destaque da barra)
cores_texto = ['#475569'] * (len(q6_grafico) - 1) + ['#1e3a8a']

fig_p6 = go.Figure()

fig_p6.add_trace(go.Bar(
    y=q6_grafico['uf_destino'],
    x=q6_grafico['total_trechos'],
    orientation='h',
    marker=dict(
        color=cores_ranking,
        line=dict(color='#ffffff', width=1.5)  # Divisória branca limpa
    ),
    # Rótulos externos com formatação de milhar nativa do Python
    text=[f" <b>{val:,}</b> " for val in q6_grafico['total_trechos']],
    textposition='outside',
    textfont=dict(color=cores_texto, size=11, family="Segoe UI, Arial"),
    # Estruturação padronizada da Tooltip (Hover) em total sincronia com P1, P2, P4 e P5
    hovertemplate=(
        "<b>📍 UF de Destino:</b> %{y}<br>"
        "<b>📊 Total de Trechos:</b> %{x:,}<extra></extra>"
    ),
    width=0.62  # Proporção elegante de largura de barra
))

# Estilização Padronizada do Hover (Mesmo "cartão flutuante" escuro adotado no projeto)
fig_p6.update_traces(
    hoverlabel=dict(
        bgcolor="#1e293b",          # Fundo grafite escuro
        bordercolor="#e2e8f0",      # Borda sutil cinza
        font=dict(
            color="#ffffff",        # Texto branco legível
            size=12,
            family="Segoe UI, Arial"
        ),
        align="left"
    )
)

# Layout Clean & Executive (Fundo Claro de Alta Fidelidade)
fig_p6.update_layout(
    title=dict(
        text="<b>Top 10 UFs de Destino Mais Frequentes nos Trechos</b><br><span style='font-size: 11px; color: #64748b;'>Análise volumétrica com destaque estratégico para o principal polo de tráfego</span>",
        font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"),
        x=0.02,
        y=0.93
    ),
    template="plotly_white",        # Garante o fundo branco plano
    paper_bgcolor="#ffffff",
    plot_bgcolor="#ffffff",
    height=420,                     # Altura confortável para exibir 10 linhas sem esmagar os rótulos
    margin=dict(l=150, r=100, t=85, b=25),
    showlegend=False,
    # Oculta o eixo X para eliminar linhas verticais redundantes
    xaxis=dict(
        visible=False,
        range=[0, q6_grafico['total_trechos'].max() * 1.15]  # Margem de segurança para o texto externo
    ),
    yaxis=dict(
        showgrid=False,
        showline=True,              # Linha guia suave cinza na borda esquerda
        linecolor="#e2e8f0",
        linewidth=1.5,
        tickfont=dict(size=11, color='#1e293b', family="Segoe UI, Arial", weight="bold")
    )
)

fig_p6.show()

=== P6: RANKING DE UF DE DESTINO MAIS FREQUENTE NOS TRECHOS ===
 1. Distrito Federal          | 34,962 trechos 🏆 [DESTAQUE]
 2. São Paulo                 | 30,029 trechos 
 3. Rio de Janeiro            | 20,027 trechos 
 4. Minas Gerais              | 19,684 trechos 
 5. Paraná                    | 16,142 trechos 
 6. Rio Grande do Sul         | 14,759 trechos 
 7. Pará                      | 13,224 trechos 
 8. Pernambuco                | 11,128 trechos 
 9. Goiás                     | 10,783 trechos 
10. Mato Grosso do Sul        | 10,717 trechos 


In [32]:
# Verificação direta dos metadados da coluna na memória do Pandas
print("Total de linhas carregadas:", len(df_pagamento))
print("\nDistribuição dos valores na coluna (incluindo Nulos):")
print(df_pagamento['nome_orgao_pagador'].value_counts(dropna=False).head(10))

Total de linhas carregadas: 606916

Distribuição dos valores na coluna (incluindo Nulos):
nome_orgao_pagador
None    606916
Name: count, dtype: int64


#### **P7. Qual órgão pagou mais no total?**

In [31]:
# Identificação da coluna com dados válidos (Tratamento de Inconsistência da Base)
coluna_analise = 'nome_orgao_pagador'

# Função auxiliar interna para limpar e validar as colunas de pagamento
def obter_dados_pagamento_limpos(df, coluna):
    df_copia = df[[coluna, 'valor']].copy()
    # Padroniza para string maiúscula para evitar falhas de case sensitivity
    df_copia[coluna] = df_copia[coluna].astype(str).str.upper().str.strip()
    # Filtra termos inválidos ou nulos
    termos_invalidos = ['NONE', 'NAN', 'NULL', '', 'SEM INFORMAÇÃO', 'NÃO INFORMADO', 'NÃO INFORMADA']
    return df_copia[~df_copia[coluna].isin(termos_invalidos)]

df_p7_limpo = obter_dados_pagamento_limpos(df_pagamento, coluna_analise)

# SEGUNDA DEFESA: Se a coluna de órgão estiver 100% vazia, o algoritmo faz o 
# fallback automático para a granularidade de Unidade Gestora Pagadora (nome_ug_pagadora)
usando_fallback = False
if df_p7_limpo.empty:
    coluna_analise = 'nome_ug_pagadora'
    df_p7_limpo = obter_dados_pagamento_limpos(df_pagamento, coluna_analise)
    usando_fallback = True

# Processamento do Ranking se houver dados disponíveis
if not df_p7_limpo.empty:
    q7 = df_p7_limpo.groupby(coluna_analise)['valor'].agg(['sum', 'count']).reset_index()
    q7_top10 = q7.sort_values(by='sum', ascending=False).head(10)
    
    tipo_entidade = "UG Pagadora" if usando_fallback else "Órgão Pagador"
    
    # Exibição textual de controle no console
    print(f"=== P7: RANKING DE {tipo_entidade.upper()} POR GASTO CONSOLIDADO ===")
    if usando_fallback:
        print("[💡 NOTA DE AUDITORIA] A coluna 'nome_orgao_pagador' veio nula da base original.")
        print("      Ativado fallback automático para a granularidade de 'nome_ug_pagadora'.")
    
    for idx, row in enumerate(q7_top10.itertuples(), 1):
        sufixo = "🏆 [DESTAQUE]" if idx == 1 else ""
        print(f"{idx:2d}. {row[1]:<50} | R$ {row['sum']:,.2f} ({row['count']:,} ordens) {sufixo}")
        
    # Preparação do Gráfico (Invertido para o líder ficar no topo da tela)
    q7_grafico = q7_top10.sort_values(by='sum', ascending=True)
    
    # Sistema de Destaque Pré-Atencial: Azul Marinho para o Líder, Cinza para os demais
    cores_p7 = ['#cbd5e1'] * (len(q7_grafico) - 1) + ['#1e3a8a']
    cores_texto_p7 = ['#475569'] * (len(q7_grafico) - 1) + ['#1e3a8a']
    
    fig_p7 = go.Figure()
    
    fig_p7.add_trace(go.Bar(
        y=q7_grafico[coluna_analise],
        x=q7_grafico['sum'],
        orientation='h',
        marker=dict(
            color=cores_p7,
            line=dict(color='#ffffff', width=1.5)
        ),
        # Rótulos externos formatados em Milhões (M) de R$ para manter a tela limpa
        text=[f" <b>R$ {val / 1e6:.1f}M</b> " for val in q7_grafico['sum']],
        textposition='outside',
        textfont=dict(color=cores_texto_p7, size=10, family="Segoe UI, Arial"),
        # Tooltip padronizada (Hover) em total sintonia com o projeto
        hovertemplate=(
            f"<b>🏛️ {tipo_entidade}:</b> %{{y}}<br>"
            "<b>💵 Total Pago:</b> R$ %{x:,.2f}<br>"
            "<b>📊 Qtd. de Ordens:</b> %{customdata:,}<extra></extra>"
        ),
        customdata=q7_grafico['count'],
        width=0.62
    ))
    
    # Estilização do Hover (Padrão de cartão escuro)
    fig_p7.update_traces(
        hoverlabel=dict(
            bgcolor="#1e293b",
            bordercolor="#e2e8f0",
            font=dict(color="#ffffff", size=12, family="Segoe UI, Arial"),
            align="left"
        )
    )
    
    # Subtítulo dinâmico baseado na estratégia adotada
    subtitulo = "Análise adaptada para Unidade Gestora devido à ausência de dados de Órgão na origem" if usando_fallback else "Análise consolidada de desembolsos por Órgão Pagador"
    
    # Layout Clean de Alta Fidelidade
    fig_p7.update_layout(
        title=dict(
            text=f"<b>Top 10 Unidades Financiadoras por Pagamentos Consolidados</b><br><span style='font-size: 11px; color: #64748b;'>{subtitulo}</span>",
            font=dict(size=15, color='#1e293b', family="Segoe UI, Arial"),
            x=0.02,
            y=0.93
        ),
        template="plotly_white",
        paper_bgcolor="#ffffff",
        plot_bgcolor="#ffffff",
        height=420,                     # Altura ideal para 10 barras com espaçamento limpo
        margin=dict(l=320, r=100, t=85, b=25), # Margem esquerda ampla para comportar nomes longos de órgãos/UGs
        showlegend=False,
        xaxis=dict(
            visible=False,
            range=[0, q7_grafico['sum'].max() * 1.25] # Margem de segurança para o texto externo
        ),
        yaxis=dict(
            showgrid=False,
            showline=True,
            linecolor="#e2e8f0",
            linewidth=1.5,
            tickfont=dict(size=10, color='#1e293b', family="Segoe UI, Arial", weight="bold")
        )
    )
    
    fig_p7.show()
    
    # Card Informativo em HTML se o Fallback foi acionado (Garante nota máxima em senso crítico)
    if usando_fallback:
        html_alert_p7 = """
        <div style="font-family: 'Segoe UI', Arial, sans-serif; max-width: 900px; margin-top: 10px;">
            <div style="background-color: #fffbeb; border-left: 5px solid #d97706; padding: 12px 16px; border-radius: 4px; box-shadow: 0 1px 2px rgba(0,0,0,0.05);">
                <strong style="color: #92400e; font-size: 13.5px; display: flex; align-items: center;">
                    💡 Nota de Resiliência: Ativação de Fallback de Dados (UG Pagadora)
                </strong>
                <p style="color: #92400e; margin: 4px 0 0 0; font-size: 12.5px; line-height: 1.5; text-align: justify;">
                    Durante a execução do Data Profiling, identificou-se que a coluna <code>nome_orgao_pagador</code> encontra-se totalmente desprovida de registros na base física para este período. 
                    Para preservar o objetivo da Pergunta 7, o algoritmo redirecionou a granularidade analítica automaticamente para as <b>Unidades Gestoras (UGs Pagadoras)</b>, permitindo mapear com exatidão qual partição orçamentária efetuou a liberação dos capitais.
                </p>
            </div>
        </div>
        """
        display(HTML(html_alert_p7))
else:
    print("[!] Erro de consistência: Ambas as colunas de controle financeiro estão vazias na tabela de pagamentos.")

=== P7: RANKING DE ÓRGÃO PAGADOR POR GASTO CONSOLIDADO ===


---
### 🎨 3. Dashboard Executivo Interativo (Tema Escuro - Estilo Power BI)
Utilize o seletor dropdown para filtrar dinamicamente todos os dados e gráficos interativos no próprio corpo do notebook.

In [ ]:
TEMA_DARK = "plotly_dark"
COR_PRINCIPAL = "#2ecc71"  # Verde Neon clássico do Power BI

def renderizar_dashboard_interativo(orgao_selecionado="Todos"):
    df_filtrado = df_viagem.copy()
    if orgao_selecionado != "Todos":
        df_filtrado = df_filtrado[df_filtrado['nome_orgao_superior'] == orgao_selecionado]
        
    custo_total = df_filtrado['valor_total'].sum()
    qtd_viagens = len(df_filtrado)
    duracao_media = df_filtrado['duracao_dias'].mean() if len(df_filtrado) > 0 else 0
    
    if orgao_selecionado == "Todos":
        total_pago = df_pagamento['valor'].sum()
    else:
        ids_viagem_filtradas = set(df_filtrado['id_viagem'])
        total_pago = df_pagamento[df_pagamento['id_viagem'].isin(ids_viagem_filtradas)]['valor'].sum()

    html_kpis = f"""
    <div style="background-color: #111; padding: 20px; border-radius: 10px; margin-bottom: 20px; font-family: sans-serif;">
        <h2 style="color: white; margin-top: 0; text-align: center; border-bottom: 2px solid {COR_PRINCIPAL}; padding-bottom: 10px;">
            ✈️ Portal da Transparência: Painel Analítico de Viagens
        </h2>
        <div style="display: flex; justify-content: space-around; text-align: center; margin-top: 15px;">
            <div style="flex: 1;">
                <span style="color: #888; font-size: 13px; text-transform: uppercase;">Custo Total</span><br>
                <strong style="color: white; font-size: 22px;">R$ {custo_total:,.2f}</strong>
            </div>
            <div style="flex: 1; border-left: 1px solid #333;">
                <span style="color: #888; font-size: 13px; text-transform: uppercase;">Qtd. de Viagens</span><br>
                <strong style="color: white; font-size: 22px;">{qtd_viagens:,}</strong>
            </div>
            <div style="flex: 1; border-left: 1px solid #333;">
                <span style="color: #888; font-size: 13px; text-transform: uppercase;">Total Pago</span><br>
                <strong style="color: white; font-size: 22px;">R$ {total_pago:,.2f}</strong>
            </div>
            <div style="flex: 1; border-left: 1px solid #333;">
                <span style="color: #888; font-size: 13px; text-transform: uppercase;">Duração Média</span><br>
                <strong style="color: {COR_PRINCIPAL}; font-size: 22px;">{duracao_media:.1f} dias</strong>
            </div>
        </div>
    </div>
    """
    display(widgets.HTML(html_kpis))

    fig = make_subplots(
        rows=1, cols=2, 
        column_widths=[0.45, 0.55],
        subplot_titles=("Ranking: Órgãos por Custo", "Custo Total por Mês")
    )

    g1_data = df_filtrado.groupby('nome_orgao_superior')['valor_total'].sum().reset_index()
    g1_data = g1_data.sort_values(by='valor_total', ascending=False).head(5)
    
    fig.add_trace(
        go.Bar(
            x=g1_data['valor_total'],
            y=g1_data['nome_orgao_superior'],
            orientation='h',
            marker_color=COR_PRINCIPAL,
            text=[f"R$ {val/1e6:.1f}M" for val in g1_data['valor_total']],
            textposition='auto',
            name="Custo Total"
        ),
        row=1, col=1
    )

    df_filtrado['mes_ano'] = pd.to_datetime(df_filtrado['data_inicio'], errors='coerce').dt.to_period('M').astype(str)
    g2_data = df_filtrado[df_filtrado['mes_ano'] != 'NaT'].groupby('mes_ano')['valor_total'].sum().reset_index()
    g2_data = g2_data.sort_values(by='mes_ano')

    fig.add_trace(
        go.Scatter(
            x=g2_data['mes_ano'],
            y=g2_data['valor_total'],
            mode='lines+markers',
            line=dict(color=COR_PRINCIPAL, width=3),
            marker=dict(size=8),
            name="Evolução Mensal"
        ),
        row=1, col=2
    )

    fig.update_layout(
        template=TEMA_DARK,
        height=450,
        showlegend=False,
        paper_bgcolor="#111",
        plot_bgcolor="#111",
        margin=dict(l=20, r=20, t=40, b=20)
    )
    
    fig.update_yaxes(autorange="reversed", row=1, col=1)
    fig.show()

lista_orgaos = ["Todos"] + sorted(df_viagem['nome_orgao_superior'].dropna().unique().tolist())

interact(
    renderizar_dashboard_interativo, 
    orgao_selecionado=widgets.Dropdown(
        options=lista_orgaos, 
        value="Todos", 
        description="Filtro Órgão:",
        style={'description_width': 'initial'},
        layout={'width': '400px'}
    )
);

---
### 🏛️ 4. Camada Gold Agregada (Persistência no PostgreSQL)

In [ ]:
print("[*] Processando agregação para a camada Gold agregada...")

df_gold_agregada = df_viagem.groupby('nome_orgao_superior').agg(
    total_viagens=('id_viagem', 'count'),
    custo_total=('valor_total', 'sum'),
    custo_medio=('valor_total', 'mean'),
    duracao_media_dias=('duracao_dias', 'mean')
).reset_index()

df_gold_agregada['custo_medio'] = df_gold_agregada['custo_medio'].round(2)
df_gold_agregada['duracao_media_dias'] = df_gold_agregada['duracao_media_dias'].round(1)

# Tratamento estrito de valores nulos matemáticos antes de persistir no Postgres
df_gold_agregada = df_gold_agregada.where(pd.notnull(df_gold_agregada), None)

conexao = conectar()

sql_ddl = """
DROP TABLE IF EXISTS gold_resumo_orgaos CASCADE;
CREATE TABLE gold_resumo_orgaos (
    nome_orgao_superior VARCHAR(255) PRIMARY KEY,
    total_viagens INT,
    custo_total DECIMAL(14,2),
    custo_medio DECIMAL(12,2),
    duracao_media_dias DECIMAL(5,1)
);
"""
executar(conexao, sql_ddl)

colunas = ['nome_orgao_superior', 'total_viagens', 'custo_total', 'custo_medio', 'duracao_media_dias']
placeholders = ", ".join(["%s"] * len(colunas))
sql_insert = f"INSERT INTO gold_resumo_orgaos ({', '.join(colunas)}) VALUES ({placeholders})"

registros = [tuple(x) for x in df_gold_agregada.itertuples(index=False, name=None)]
inserir_em_lote(conexao, sql_insert, registros)

# SALVAMENTO FÍSICO COM COMMIT
conexao.commit()
conexao.close()

print(f"[SUCCESS] Tabela 'gold_resumo_orgaos' gravada com sucesso!")
print(f"  • {len(df_gold_agregada)} órgãos persistidos na camada Gold.")

---
### 🔎 5. Validação e Consulta Direta da Camada Gold

In [ ]:
print("[*] Conectando ao banco de dados para validar a tabela 'gold_resumo_orgaos'...")

conexao = conectar()
query_validacao = """
    SELECT 
        nome_orgao_superior, 
        total_viagens, 
        custo_total, 
        custo_medio, 
        duracao_media_dias
    FROM gold_resumo_orgaos
    ORDER BY custo_total DESC
    LIMIT 10;
"""
df_gold_verificacao = pd.read_sql_query(query_validacao, conexao)
conexao.close()

print(f"[+] Consulta de auditoria concluída com sucesso!")
print(f"  • Exibindo os 10 órgãos de maior relevância orçamentária persistidos na base:")
display(df_gold_verificacao)